In [1]:
!pip -q install transformers torch pandas matplotlib

In [2]:
import re
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

pd.set_option("display.max_colwidth", 80)
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "gpt2"   # try "gpt2-medium" or "gpt2-large" to compare
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device).eval()

print(f"Loaded {MODEL_NAME} on {device} | vocabulary size: {tokenizer.vocab_size}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2 on cuda | vocabulary size: 50257


In [3]:
TITLE = "The Snow Man"
AUTHOR = "Wallace Stevens (1879-1955)"

POEM = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

In [4]:
LINE_RE = re.compile(r"^(?P<prefix>.*?)(?P<word>[A-Za-z][A-Za-z'\-]*)(?P<punct>[^A-Za-z]*)$")

def split_last_word(line):
    """Return (prefix, last_word, trailing_punctuation)."""
    m = LINE_RE.match(line)
    if not m:
        return line, "", ""
    return m["prefix"], m["word"], m["punct"]

for l in POEM.splitlines()[:4]:
    print(split_last_word(l))

('One must have a mind of ', 'winter', '')
('To regard the frost and the ', 'boughs', '')
('Of the pine-trees crusted with ', 'snow', ';')
('And have been cold a long ', 'time', '')


In [5]:
@torch.no_grad()
def next_token_candidates(prompt, top_k=500):
    """Return [(token_string, probability), ...] sorted from most to least likely."""
    if not prompt:
        prompt = tokenizer.bos_token          # GPT-2 needs at least one token
    ids = tokenizer(prompt, return_tensors="pt").input_ids[:, -1024:].to(device)
    logits = model(ids).logits[0, -1]
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(probs, top_k)
    return [(tokenizer.decode([i]), p) for i, p in zip(top.indices.tolist(), top.values.tolist())]


def is_word_token(tok, leading_space=True):
    """A 'word' token: optional leading space followed only by letters."""
    if leading_space:
        return tok.startswith(" ") and tok[1:].isalpha()
    return tok.isalpha()


@torch.no_grad()
def complete_word(prompt, first_piece, max_extra=3):
    """Optionally extend a word-start token greedily ('glitt' -> 'glitter')."""
    word = first_piece
    for _ in range(max_extra):
        nxt, _ = next_token_candidates(prompt + word, top_k=1)[0]
        if nxt.isalpha() and not nxt.startswith(" "):
            word += nxt
        else:
            break
    return word


def pick_rank(prompt, x, filter_words=True, leading_space=True,
              exclude=None, finish_word=False):
    """Return (word, probability, raw_rank) for the x-th ranked candidate (1-indexed)."""
    top_k = max(500, x * 30)
    count = 0
    for raw_rank, (tok, p) in enumerate(next_token_candidates(prompt, top_k), start=1):
        if filter_words and not is_word_token(tok, leading_space):
            continue
        if exclude and tok.strip().lower() == exclude.lower():
            continue
        count += 1
        if count == x:
            word = complete_word(prompt, tok) if finish_word else tok
            return word.strip(), p, raw_rank
    raise ValueError(f"Fewer than {x} valid candidates in top {top_k}; raise top_k.")

In [6]:
def p_plus_x(poem, x=7, context="poem", cascade=False, filter_words=True,
             exclude_original=False, finish_word=False, preamble=""):
    lines = poem.strip("\n").split("\n")
    out_lines, history, records = [], [], []

    for n, line in enumerate(lines, start=1):
        if not line.strip():                     # keep stanza breaks
            out_lines.append(line); history.append(line); continue

        prefix, word, punct = split_last_word(line)

        ctx = preamble
        if context == "poem" and history:
            ctx += "\n".join(history) + "\n"
        left = prefix.rstrip()
        prompt = ctx + left
        leading_space = bool(left)               # a word mid-line starts with a space token

        new_word, prob, raw_rank = pick_rank(
            prompt, x, filter_words=filter_words, leading_space=leading_space,
            exclude=word if exclude_original else None, finish_word=finish_word)

        new_line = prefix + new_word + punct
        out_lines.append(new_line)
        history.append(new_line if cascade else line)
        records.append({"line": n, "original": word, f"P+{x}": new_word,
                        "prob": round(prob, 5), "raw_rank": raw_rank})

    return "\n".join(out_lines), pd.DataFrame(records)


def save_poem(text, x, folder="."):
    path = f"{folder}/P+{x}.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"{TITLE} (P+{x})\nafter {AUTHOR}\n\n{text}\n")
    print("Saved", path)
    return path

In [7]:
p7_text, p7_table = p_plus_x(POEM, x=7, context="poem", cascade=False)
print(p7_text)
p7_table

One must have a mind of her
To regard the frost and the ice
Of the pine-trees crusted with blood;
And have been cold a long long
To behold the junipers shagged with blood,
The spruces rough in the distant forest
Of the January sun; and not to know
Of any misery in the sound of the thunder,
In the sound of a few birds,
Which is the sound of the summer
Full of the same air
That is blowing in the same bare space
For the listener, who listens in the sound,
And, nothing himself, does
Nothing that is not there and the nothing that exists.


,line,original,P+7,prob,raw_rank
0,1,winter,her,0.01623,7
1,2,boughs,ice,0.01349,7
2,3,snow,blood,0.01536,7
3,4,time,long,0.00832,8
4,5,ice,blood,0.01111,7
5,6,glitter,forest,0.01320,9
6,7,think,know,0.02101,7
7,8,wind,thunder,0.00959,7
8,9,leaves,birds,0.00881,7
9,10,land,summer,0.00956,7


In [8]:
save_poem(p7_text, 7)

Saved ./P+7.txt


'./P+7.txt'

In [9]:
X_VALUES = [1, 2, 3, 5, 7, 10, 15, 25, 50, 100, 250]

versions, comparison = {}, pd.DataFrame()
for x in X_VALUES:
    text, table = p_plus_x(POEM, x=x)
    versions[x] = text
    if comparison.empty:
        comparison["original"] = table["original"]
    comparison[f"P+{x}"] = table[f"P+{x}"]

comparison   # one column per x: the last word of each line

,original,P+1,P+2,P+3,P+5,P+7,P+10,P+15,P+25,P+50,P+100,P+250
0,winter,their,its,his,a,her,one,some,mystery,stone,ours,principle
1,boughs,cold,snow,rain,heat,ice,hail,fall,weather,black,ground,e
2,snow,snow,the,moss,a,blood,gold,th,dead,water,o,lo
3,time,time,while,winter,year,long,night,journey,period,s,months,record
4,ice,snow,the,their,ice,blood,leaves,moss,g,grass,t,horn
5,glitter,woods,hills,trees,snow,forest,valleys,wind,gro,w,me,to
6,think,be,see,look,fear,know,say,mention,remember,return,lay,change
7,wind,wind,th,spring,snow,thunder,waves,distant,drum,h,high,be
8,leaves,th,of,voices,p,birds,sigh,whist,a,g,bell,young
9,land,wind,spring,snow,winter,summer,earth,tree,fall,stream,rose,flock


In [10]:
# Read full versions side by side
for x in X_VALUES:
    print(f"========== P+{x} ==========")
    print(versions[x], "\n")

========== P+1 ==========
One must have a mind of their
To regard the frost and the cold
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with snow,
The spruces rough in the distant woods
Of the January sun; and not to be
Of any misery in the sound of the wind,
In the sound of a few th,
Which is the sound of the wind
Full of the same sound
That is blowing in the same bare land
For the listener, who listens in the silence,
And, nothing himself, is
Nothing that is not there and the nothing that is. 

========== P+2 ==========
One must have a mind of its
To regard the frost and the snow
Of the pine-trees crusted with the;
And have been cold a long while
To behold the junipers shagged with the,
The spruces rough in the distant hills
Of the January sun; and not to see
Of any misery in the sound of the th,
In the sound of a few of,
Which is the sound of the spring
Full of the same snow
That is blowing in the same bare air
For the listener, wh

In [11]:
# Compare context settings for one x (useful material for the reflection)
x = 7
for label, kwargs in {
    "line only":         dict(context="line"),
    "poem context":      dict(context="poem"),
    "poem + cascade":    dict(context="poem", cascade=True),
}.items():
    print(f"---- P+{x} | {label} ----")
    print(p_plus_x(POEM, x=x, **kwargs)[0], "\n")

---- P+7 | line only ----
One must have a mind of her
To regard the frost and the death
Of the pine-trees crusted with oil;
And have been cold a long few
To behold the junipers shagged with white,
The spruces rough in the distant night
Of the January sun; and not to have
Of any misery in the sound of the sound,
In the sound of a few shots,
Which is the sound of the voice
Full of the same story
That is blowing in the same bare head
For the listener, who listens in the following,
And, nothing himself, I
Nothing that is not there and the nothing that isn. 

---- P+7 | poem context ----
One must have a mind of her
To regard the frost and the ice
Of the pine-trees crusted with blood;
And have been cold a long long
To behold the junipers shagged with blood,
The spruces rough in the distant forest
Of the January sun; and not to know
Of any misery in the sound of the thunder,
In the sound of a few birds,
Which is the sound of the summer
Full of the same air
That is blowing in the same bare spa

In [12]:
CHOSEN_X = 23   # TODO: replace with the value that gave the most absurd / witty result
CHOSEN_SETTINGS = dict(context="poem", cascade=False)   # TODO: record the settings you used

px_text, px_table = p_plus_x(POEM, x=CHOSEN_X, **CHOSEN_SETTINGS)
print(px_text)
save_poem(px_text, CHOSEN_X)
px_table

One must have a mind of humility
To regard the frost and the dead
Of the pine-trees crusted with green;
And have been cold a long fortnight
To behold the junipers shagged with red,
The spruces rough in the distant branches
Of the January sun; and not to sleep
Of any misery in the sound of the b,
In the sound of a few snow,
Which is the sound of the air
Full of the same thing
That is blowing in the same bare heaven
For the listener, who listens in the gloom,
And, nothing himself, that
Nothing that is not there and the nothing that passes.
Saved ./P+23.txt


,line,original,P+23,prob,raw_rank
0,1,winter,humility,0.00229,25
1,2,boughs,dead,0.00442,23
2,3,snow,green,0.00402,23
3,4,time,fortnight,0.00098,25
4,5,ice,red,0.00379,23
5,6,glitter,branches,0.00474,25
6,7,think,sleep,0.00538,23
7,8,wind,b,0.00420,23
8,9,leaves,snow,0.00424,24
9,10,land,air,0.00466,23


In [13]:
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")


def noun_p_plus_x(poem, x=7, match_tag=True, top_k=600):
    lines = poem.strip("\n").split("\n")
    out_lines, records = [], []

    for n, line in enumerate(lines, start=1):
        new_line, shift = line, 0
        for tok in nlp(line):
            if tok.pos_ != "NOUN":
                continue
            start = tok.idx + shift
            end = start + len(tok.text)
            left, right = new_line[:start], new_line[end:]
            ctx = "\n".join(out_lines) + "\n" if out_lines else ""
            prompt = ctx + left.rstrip()
            leading_space = bool(left.strip())

            accepted, chosen = 0, None
            for cand, p in next_token_candidates(prompt, top_k):
                if not is_word_token(cand, leading_space):
                    continue
                w = cand.strip()
                span = nlp(left + w + right).char_span(start, start + len(w))
                if span is None or len(span) != 1:
                    continue
                t = span[0]
                if t.pos_ == "NOUN" and (not match_tag or t.tag_ == tok.tag_):
                    accepted += 1
                    if accepted == x:
                        chosen = (w, p)
                        break

            if chosen:
                w, p = chosen
                new_line = left + w + right
                shift += len(w) - len(tok.text)
                records.append({"line": n, "noun": tok.text, f"N/P+{x}": w, "prob": round(p, 5)})
        out_lines.append(new_line)

    return "\n".join(out_lines), pd.DataFrame(records)


nouns_text, nouns_table = noun_p_plus_x(POEM, x=7)
print(nouns_text)
nouns_table

One must have a bit of knowledge
To regard the history and the methods
Of the work-trees crusted with dirt;
And have been cold a long winter
To behold the things shagged with rust,
The walls rough in the distant sun
Of the January morning; and not to think
Of any part in the day of the moon,
In the midst of a few stars,
Which is the moon of the east
Full of the same color
That is blowing in the same bare earth
For the night, who listens in the light,
And, nothing himself, beholds
Nothing that is not there and the nothing that is.


,line,noun,N/P+7,prob
0,1,mind,bit,0.00660
1,1,winter,knowledge,0.00871
2,2,frost,history,0.00441
3,2,boughs,methods,0.00363
4,3,pine,work,0.00565
5,3,snow,dirt,0.00604
6,4,time,winter,0.00210
7,5,junipers,things,0.00182
8,5,ice,rust,0.00596
9,6,spruces,walls,0.00229
